# Notebook 04 — Modeling & Evaluation

In this notebook, we build, train, and evaluate multiple machine learning models
to predict **annual apartment rental prices in JVC, Dubai**.

The workflow follows a standard and production-ready machine learning pipeline:

1. Load and prepare the dataset
2. Define features (X) and target (y)
3. Split data into training and testing sets
4. Establish a baseline model
5. Train and evaluate multiple regression models
6. Compare model performance using MAE, RMSE, and R²
7. Select the best-performing model
8. Improve it using hyperparameter tuning
9. Evaluate the tuned model on unseen data
10. Save the final model for future predictions

All models are evaluated on the **same test set** to ensure a fair comparison.

## Imports

In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

from xgboost import XGBRegressor
import lightgbm as lgb

import joblib  # to save the model



from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Load dataset

The csv we prepared in step 03.

In [13]:
DATA_PATH = "data/jvc_apartments_ml.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (951, 51)


,title,price,frequency,bedrooms,bathrooms,area,location,url,price_clean,price_yearly_aed,...,building_grouped_Westview Garden,district_JVC District 11,district_JVC District 12,district_JVC District 13,district_JVC District 14,district_JVC District 15,district_JVC District 16,district_JVC District 17,district_JVC District 18,district_Unknown
0,1BR Apartment for Rent | Balcony & Pool | JVC,"74,000",yearly,1,2,905 sqft,"AAA Residence, JVC District 13, Jumeirah Villa...",https://www.bayut.com/property/details-9398172...,74000,74000,...,False,False,False,True,False,False,False,False,False,False
1,1 B/R with Balcony | Pool & Gym | JVC,"69,000",yearly,1,1,883 sqft,"Emerald Tower, JVC District 18, Jumeirah Villa...",https://www.bayut.com/property/details-4864285...,69000,69000,...,False,False,False,False,False,False,False,False,True,False
2,"Binghatti Phoenix, Jumeirah Village Circle, Dubai","89,990",yearly,1,2,826 sqft,"Binghatti Phoenix, JVC District 13, Jumeirah V...",https://www.bayut.com/property/details-1347054...,89990,89990,...,False,False,False,True,False,False,False,False,False,False
3,Converted into 2BR | Private Garden | Furnished,"140,000",yearly,1,2,"1,133 sqft","Signature Livings South, Signature Livings, JV...",https://www.bayut.com/property/details-1330702...,140000,140000,...,False,False,False,False,False,False,False,False,False,False
4,Spacious 1Br | Prime Location | JVC,"75,000",yearly,1,2,925 sqft,"Reef Residence, JVC District 13, Jumeirah Vill...",https://www.bayut.com/property/details-1366409...,75000,75000,...,False,False,False,True,False,False,False,False,False,False


In [14]:
print(df.columns)

Index(['title', 'price', 'frequency', 'bedrooms', 'bathrooms', 'area',
       'location', 'url', 'price_clean', 'price_yearly_aed', 'bedrooms_clean',
       'bathrooms_clean', 'area_clean', 'area_per_bedroom',
       'bathrooms_per_bedroom', 'log_area', 'log_price', 'building',
       'community', 'property_type_penthouse', 'property_type_townhouse',
       'property_type_villa', 'building_grouped_Binghatti Corner',
       'building_grouped_Binghatti Crest',
       'building_grouped_Binghatti Heights',
       'building_grouped_Binghatti House',
       'building_grouped_Binghatti Phantom',
       'building_grouped_Binghatti Phoenix',
       'building_grouped_Binghatti Royale',
       'building_grouped_Bloom Heights 1, Bloom Heights',
       'building_grouped_DAMAC Ghalia', 'building_grouped_Fortunato',
       'building_grouped_Imperial Tower', 'building_grouped_Laya Residences',
       'building_grouped_Other', 'building_grouped_Pearl House 2',
       'building_grouped_Reef Residence', 

## Define Train (X) and Target (y) features

In [15]:
# We must drop columns that are useless (memory usage) and derived from the `price_yearly_aed` (data leakage)
# this step is not cleaning or preprocessesing, this is why I put it here. In ML we now choose only what we need.
# also after we encoded, all the other non-numeric features are dropped because many models cannot handle strings
DROP_COLS = [
    "title",
    "price",      # Derived features cause data leakage
    "price_clean", # Derived
    "frequency",
    "url",
    "location",
    "area",
    "bedrooms",
    "bathrooms",
    "building",
    "community",

    "price_yearly_aed",
    "log_price" # Derived

]

X = df.drop(columns=[c for c in DROP_COLS if c in df.columns]) # Here we drop the usless or derived features

# Sanitize column names for LightGBM Model (remove spaces, commas, special chars)
X.columns = X.columns.str.replace(r"[ ,]", "_", regex=True)

y = df["price_yearly_aed"] # TARGET

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (951, 38)
y shape: (951,)


In [16]:
print(X.columns)
y.head()

Index(['bedrooms_clean', 'bathrooms_clean', 'area_clean', 'area_per_bedroom',
       'bathrooms_per_bedroom', 'log_area', 'property_type_penthouse',
       'property_type_townhouse', 'property_type_villa',
       'building_grouped_Binghatti_Corner', 'building_grouped_Binghatti_Crest',
       'building_grouped_Binghatti_Heights',
       'building_grouped_Binghatti_House',
       'building_grouped_Binghatti_Phantom',
       'building_grouped_Binghatti_Phoenix',
       'building_grouped_Binghatti_Royale',
       'building_grouped_Bloom_Heights_1__Bloom_Heights',
       'building_grouped_DAMAC_Ghalia', 'building_grouped_Fortunato',
       'building_grouped_Imperial_Tower', 'building_grouped_Laya_Residences',
       'building_grouped_Other', 'building_grouped_Pearl_House_2',
       'building_grouped_Reef_Residence', 'building_grouped_Rigel_Apartments',
       'building_grouped_Rose_10', 'building_grouped_SH_Living_1',
       'building_grouped_Sydney_Villas', 'building_grouped_Westview_Garde

,price_yearly_aed
0,74000
1,69000
2,89990
3,140000
4,75000


## Test/Train Split

Here is where we will split the data. We will then train the models on one part (80%) and the rest is for testing the model.

```
TRAIN / TEST SPLIT
   │
   ├──► X_train, y_train  ──► MODEL TRAINING
   │
   └──► X_test, y_test    ──► MODEL EVALUATION
                                │
                                ▼
                          METRICS (MAE, RMSE, R²)
                                │
                                ▼
                        MODEL COMPARISON
                                │
                                ▼
                       BEST MODEL SELECTED
                                │
                                ▼
                    SAVE MODEL / USE FOR PREDICTION
```

_The model learns from (X_train, y_train), predicts using X_test, and we evaluate by comparing predictions to y_test._

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42 # controls randomness, and keeps results reproducible
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)




Train shape: (760, 38) (760,)
Test shape: (191, 38) (191,)


## Baseline model

we create a baseline model to check how a simple baseline performs.    
Metrics used to evaluate this model is the `MAE`, `RMSE`, and `R2`


In [18]:
# Baseline predictions, no training happens in baseline model
y_pred_baseline = np.full_like(y_test, y_train.mean(), dtype=np.float64)

mae_baseline = mean_absolute_error(y_test, y_pred_baseline)
mse_baseline = mean_squared_error(y_test, y_pred_baseline)
rmse_baseline = np.sqrt(mse_baseline)
r2_baseline = r2_score(y_test, y_pred_baseline)

print("Baseline:")
print(f"MAE: {mae_baseline:,.0f} AED")
print(f"RMSE: {rmse_baseline:,.0f} AED")
print(f"R²: {r2_baseline:.2f}")

Baseline:
MAE: 33,740 AED
RMSE: 48,103 AED
R²: -0.00


Understanding the metrics:

So for our case, we will try to predic yearly rent, which is a number. The model makes guesses, Then we will have to check how well it guesses...     

     
`MAE`: On avg, how far is the guess from the real number?
- Formula: Take the difference between prediction and real value, make it positive, average it.
- ```
    Example:
    Real rents = [1000, 2000, 3000]
    Predicted = [1200, 1800, 3100]
    Errors = [200, 200, 100] → MAE = (200+200+100)/3 = 167
    ```
- LOWER is better

    

`RSME`: Similar to `MAE`, but we square the errors first, then take square root.
- Big mistakes are punished more.
- ```
    Example:
    Errors = [200, 200, 100]
    Squared = [40000, 40000, 10000] → mean = 30000 → sqrt = 173.2
    ```
- LOWER is better

   
`R^2`: How much better is the model than just guessing the mean?
- ```
    R² = 0 → model explains 0% of the variation (just guessing the mean)
    R² = 0.7 → model explains 70% of why rents differ
    R² = 1 → perfect model
    R² < 0 → your model is worse than guessing the mean
    ```

- HIGHER is better (closer to 1)




## ML Models

In this section I will train and evaluate 5 ML models and then choose the best.

### Linear Regression (LR)

_Linear Regression is like drawing a straight line through your data to predict a number._

In [19]:
# Initialize
lr = LinearRegression()

# Train the model
lr.fit(X_train, y_train)

# Predict on test set
y_pred_lr = lr.predict(X_test)

# Evaluate
mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))  # sqrt of MSE
r2_lr = r2_score(y_test, y_pred_lr)

print("Linear Regression Performance:")
print(f"MAE: {mae_lr:,.0f} AED")
print(f"RMSE: {rmse_lr:,.0f} AED")
print(f"R²: {r2_lr:.2f}")

Linear Regression Performance:
MAE: 13,491 AED
RMSE: 21,162 AED
R²: 0.81


### Tree-based Models

Now we try non-linear models, the trees.

#### Decision Tree Regressor

In [20]:
# Initialize model
dt = DecisionTreeRegressor(random_state=42, max_depth=5)  # limit depth to prevent overfitting

# Train
dt.fit(X_train, y_train)

# Predict
y_pred_dt = dt.predict(X_test)

# Evaluate
mae_dt = mean_absolute_error(y_test, y_pred_dt)
rmse_dt = np.sqrt(mean_squared_error(y_test, y_pred_dt))
r2_dt = r2_score(y_test, y_pred_dt)

print(f"Decision Tree Performance:\nMAE: {mae_dt:,.0f} AED\nRMSE: {rmse_dt:,.0f} AED\nR²: {r2_dt:.2f}")

Decision Tree Performance:
MAE: 14,179 AED
RMSE: 21,159 AED
R²: 0.81


#### Random Forest Regressor


Better than normal Decision Tree beacues it reduces overfitting (when model memorizes the data instead of the general rule) and bias.

In [27]:
# Initialize
rf = RandomForestRegressor(n_estimators=100, random_state=42)

# Train
rf.fit(X_train, y_train)

# Predict
y_pred_rf = rf.predict(X_test)

# Evaluate
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Random Forest Performance:\nMAE: {mae_rf:,.0f} AED\nRMSE: {rmse_rf:,.0f} AED\nR²: {r2_rf:.2f}")


Random Forest Performance:
MAE: 12,056 AED
RMSE: 19,361 AED
R²: 0.84


### Gradient Boosting Models

#### XGBoost

This is basically a Random Forest regressor, but each new tree tries to fix the mistakes of the previous one.

In [28]:
# Initialize model
xgb = XGBRegressor(
    n_estimators=200,      # number of trees
    max_depth=5,           # depth of each tree
    learning_rate=0.1,     # step size
    random_state=42
)

# Train
xgb.fit(X_train, y_train)

# Predict
y_pred_xgb = xgb.predict(X_test)

# Evaluate
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)

print(f"XGBoost Performance:\nMAE: {mae_xgb:,.0f} AED\nRMSE: {rmse_xgb:,.0f} AED\nR²: {r2_xgb:.2f}")

XGBoost Performance:
MAE: 12,182 AED
RMSE: 19,598 AED
R²: 0.83


#### LightGBM

This is like the XGBoost, but this one is optimized for speed and memory usage.

In [29]:
# Initialize LightGBM Regressor
lgbm = lgb.LGBMRegressor(random_state=42, n_estimators=500, learning_rate=0.05, verbose=-1)

# Train
lgbm.fit(X_train, y_train)

# Predict
y_pred_lgbm = lgbm.predict(X_test)

# Evaluate
mae_lgbm = mean_absolute_error(y_test, y_pred_lgbm)
rmse_lgbm = np.sqrt(mean_squared_error(y_test, y_pred_lgbm))
r2_lgbm = r2_score(y_test, y_pred_lgbm)

print(f"LightGBM Performance:\nMAE: {mae_lgbm:,.0f} AED\nRMSE: {rmse_lgbm:,.0f} AED\nR²: {r2_lgbm:.2f}")

LightGBM Performance:
MAE: 12,817 AED
RMSE: 20,226 AED
R²: 0.82


## Evaluation

In [30]:
results = {
    "Baseline":      {"MAE": mae_baseline, "RMSE": rmse_baseline, "R2": r2_baseline},
    "Linear Regression": {"MAE": mae_lr, "RMSE": rmse_lr, "R2": r2_lr},
    "Decision Tree": {"MAE": mae_dt, "RMSE": rmse_dt, "R2": r2_dt},
    "Random Forest": {"MAE": mae_rf, "RMSE": rmse_rf, "R2": r2_rf},
    "XGBoost":       {"MAE": mae_xgb, "RMSE": rmse_xgb, "R2": r2_xgb},
    "LightGBM":      {"MAE": mae_lgbm, "RMSE": rmse_lgbm, "R2": r2_lgbm},
}

# Convert to DataFrame for easier display
df_results = pd.DataFrame(results).T  # Transpose so models are rows
df_results = df_results.sort_values(by="R2", ascending=False)  # Sort by R² (higher is better)

print("Model Comparison (sorted by R², high → low):\n")
print(df_results)

# Decide “best” model:
# highest R² *and* lowest errors (MAE & RMSE).
best_by_r2 = df_results["R2"].idxmax()
best_by_mae = df_results["MAE"].idxmin()
best_by_rmse = df_results["RMSE"].idxmin()

print("\nBest by R² (explains most variation):", best_by_r2)
print("Best by MAE (lowest average error):", best_by_mae)
print("Best by RMSE (lowest large errors):", best_by_rmse)

# A simple rule: choose model that appears most often in the above metrics
winners = [best_by_r2, best_by_mae, best_by_rmse]
best_model = pd.Series(winners).mode().iloc[0]  # most frequent
print(f"\n  Overall best model: {best_model}")


Model Comparison (sorted by R², high → low):

                            MAE          RMSE        R2
Random Forest      12056.480205  19360.661002  0.838008
XGBoost            12181.881836  19598.168894  0.834009
LightGBM           12816.670471  20226.128636  0.823201
Decision Tree      14179.369701  21158.578338  0.806524
Linear Regression  13490.516220  21162.453119  0.806453
Baseline           33740.494489  48103.221521 -0.000004

Best by R² (explains most variation): Random Forest
Best by MAE (lowest average error): Random Forest
Best by RMSE (lowest large errors): Random Forest

  Overall best model: Random Forest


## Hyperparameter Tuning

Since we found what is the best model, we try to make it even better by hyperparameter tuning

In [31]:
# Imports specific to this notebook's workflow
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor


# 1. Define the model
rf = RandomForestRegressor(random_state=42)

# 2. Define the hyperparameter space
param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [None, 5, 10, 15, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None],
}

# 3. Initialize RandomizedSearchCV
rf_random = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=20,
    scoring="r2",
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# 4. Fit
rf_random.fit(X_train, y_train)

# 5. Best parameters
print("Best hyperparameters:", rf_random.best_params_)
print("Best CV R²:", rf_random.best_score_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best hyperparameters: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': None}
Best CV R²: 0.8202737264779374


## Final Evaluation + Saving

Now we comapre the tuned model to the original model with the default and see which performed better then we save it.

In [34]:

# Evaluate the tuned RF on the test set
y_pred_rf_tuned = rf_random.best_estimator_.predict(X_test)
r2_rf_tuned = r2_score(y_test, y_pred_rf_tuned)

# Compare
if r2_rf_tuned > r2_rf:
    best_model = rf_random.best_estimator_
    print(f"Hypertuned RF is better! R² = {r2_rf_tuned:.3f}")
else:
    best_model = rf
    print(f"Default RF is better! R² = {r2_rf:.3f}")

# Save the best model
joblib.dump(best_model, "models/best_model.pkl")
print("Best model saved as 'best_model.pkl'")


Hypertuned RF is better! R² = 0.844
Best model saved as 'best_model.pkl'


# Conclusion

In this notebook, we trained and evaluated several regression models to predict
annual rental prices for apartments in JVC, Dubai.

## Key findings:
- The **baseline model** performed poorly, confirming that meaningful patterns exist in the data.
- **Linear Regression** and **Decision Trees** captured some structure but were limited in performance.
- **Ensemble models** (Random Forest, XGBoost, LightGBM) significantly outperformed simpler models.
- Among all models, **Random Forest** achieved the best overall balance of:
  - High R² (explained variance)
  - Low MAE and RMSE (prediction error)

## Model optimization:
- We applied **hyperparameter tuning** to the Random Forest model using cross-validation.
- The tuned model achieved a higher R² on the test set than the default configuration.
- This confirms that careful tuning can improve model generalization.

## Final result:
- The best-performing model was saved as `best_model.pkl`
- This model can now be reused for predictions without retraining.

This concludes the modeling phase of the project.
